In [ ]:
# Quant data policy: this repo reads Google Drive data; downloads run from note.
import sys
from pathlib import Path
_QUANT_ROOT = Path("/Users/xinc/GitHub/Quant")
if str(_QUANT_ROOT) not in sys.path:
    sys.path.insert(0, str(_QUANT_ROOT))
import cloud_data


In [1]:
from pybit.unified_trading import HTTP
import json
import time
import numpy as np
import pandas as pd
from datetime import datetime
import sqlite3
import requests
import os
from pycoingecko import CoinGeckoAPI
from tqdm import tqdm
import sys
from pathlib import Path
sys.path.insert(0, "/Users/xinc/GitHub/Quant")
from cloud_data import CRYPTO_COINTEGRATION


# Bybit 資料下載

## 期現貨歷史資料下載

In [ ]:
def _local_symbols(directory):
    return sorted(path.stem for path in Path(directory).glob("*.parquet"))

def get_all_bybit_usdt_spot_symbols():
    return _local_symbols(CRYPTO_SPOT)

def get_price(symbols, categories, start_date, end_date, database, interval=1):
    symbols = symbols or get_all_bybit_usdt_spot_symbols()
    categories = categories or ["spot", "futures"]
    roots = {"spot": CRYPTO_SPOT, "linear": CRYPTO_FUTURES, "futures": CRYPTO_FUTURES}
    result = {}
    for symbol in symbols:
        for category in categories:
            path = roots[category] / f"{symbol}.parquet"
            if not path.exists():
                continue
            frame = pd.read_parquet(path)
            time_col = "open_time" if "open_time" in frame.columns else frame.columns[0]
            frame[time_col] = pd.to_datetime(frame[time_col])
            result[(symbol, category)] = frame.loc[
                frame[time_col].between(start_date, end_date)
            ].copy()
    return result


## 資金費率下載

In [ ]:
def get_all_bybit_perp_symbols():
    return _local_symbols(CRYPTO_FUTURES)

def get_funding_rate(symbols, start_date, end_date, database):
    symbols = symbols or get_all_bybit_perp_symbols()
    result = {}
    for symbol in symbols:
        path = CRYPTO_FUNDING_RATE / f"{symbol}.parquet"
        if not path.exists():
            continue
        frame = pd.read_parquet(path)
        date_col = next(
            (column for column in ("fundingTime", "timestamp", "date") if column in frame.columns),
            frame.columns[0],
        )
        frame[date_col] = pd.to_datetime(frame[date_col])
        result[symbol] = frame.loc[
            frame[date_col].between(start_date, end_date)
        ].copy()
    return result


## 板塊分類

In [ ]:
def fetch_sector_map(base_path):
    """
    建立 sector_map.json
    
    CoinGecko 的分類 API，將未分類的標的歸入 Others
    """

    categories_to_process = ["spot", "linear", "funding_rate"]
    meta_dir = os.path.join(base_path, "metadata")
    os.makedirs(meta_dir, exist_ok=True)

    cg = CoinGeckoAPI()

    for category_type in categories_to_process:
        category_dir = os.path.join(base_path, category_type)
        if not os.path.exists(category_dir):
            print(f"⚠ {category_type} 資料夾不存在：{category_dir}")
            continue

        existing_symbols = set(f.replace(".csv", "") for f in os.listdir(category_dir) if f.endswith("USDT.csv"))
        if not existing_symbols:
            print(f"⚠ {category_type} 資料夾中沒有任何 USDT 幣種資料")
            continue

        try:
            categories = cg.get_coins_categories()
        except Exception as e:
            print(f"⚠ 無法取得 CoinGecko 分類資料：{e}")
            return

        sector_map = {}

        for category in tqdm(categories, desc = f"{category_type} 分類處理中"):
            cat_id = category['id']
            cat_name = category['name']
            try:
                coins = cg.get_coins_markets(vs_currency = 'usd', category = cat_id)
                symbols = [coin['symbol'].upper() + "USDT" for coin in coins]
                matched = list(set(symbols) & existing_symbols)
                if matched:
                    sector_map[cat_name] = {
                        "id": cat_id,
                        "symbols": sorted(matched)
                    }
                time.sleep(1.2)
            except Exception as e:
                print(f"無法抓取分類 {cat_name}: {e}")

        categorized = set(sym for v in sector_map.values() for sym in v["symbols"])
        uncategorized = sorted(list(existing_symbols - categorized))

        if uncategorized:
            sector_map["Others"] = {
                "id": "uncategorized",
                "symbols": uncategorized
            }

        sector_file = os.path.join(meta_dir, f"{category_type}_sector_map.json")
        with open(sector_file, "w") as f:
            json.dump(sector_map, f, indent = 2)

In [ ]:
start_date = datetime(2022, 1, 1)
end_date = datetime.now()

symbols = []
categories = []
interval = 60
base = CRYPTO_COINTEGRATION

funding_data = get_funding_rate(symbols, start_date, end_date, base)
print(f"Loaded {len(funding_data)} local funding-rate datasets.")


# 學長code

#### Binance期貨資料下載

In [ ]:
all_symbols = ['LINKUSDT','FTMUSDT','RUNEUSDT','UNIUSDT','GRTUSDT',
               'RSRUSDT','TRBUSDT','MKBUSDT','AAVEUSDT','THETAUSDT',
               'CRVUSDT','RAYUSDT','ZRXUSDT','SNXUSDT','XTZUSDT',]

for ff in all_symbols:
    # config_logging(logging, logging.DEBUG)
    Futures_client = API()

    start_time_date = datetime(2025, 1, 1)
    end_time_date = datetime().now()
    klines_raw = []

    while start_time_date < end_time_date:
        start_time_ts = int(start_time_date.timestamp()) * 1000
        klines = Futures_client.futures_klines(symbol = ff, interval = "4h", startTime = start_time_ts, limit = 1000)
        klines_raw.extend(klines)
        last_klines = klines[-1][0]
        start_time_date = last_klines + 1
        time.sleep(15)

    # preprocessing 1
    headers = ['open_time', 'open', 'high', 'low', 'close', 'volume', 'close_time', 'quote_vol', '#trade', 'taker_buy_vol', 'taker_buy_quote_vol', 'ignore']
    klines_df = pd.DataFrame(klines_raw, columns = headers, dtype = float)
    klines_df.drop(columns = ['ignore', 'quote_vol', 'taker_buy_quote_vol'], inplace = True)

    # Convert timestamps to datetime objects
    klines_df['open_time'] = list(map(lambda x: datetime.fromtimestamp(x/1000),klines_df['open_time']))
    klines_df['close_time'] = list(map(lambda x: datetime.fromtimestamp(x/1000),klines_df['close_time']))
    # preprocessing 4
    first_day_of_week = 3 # the day of the week of the very first day of the interval
    week_list = list((klines_df.index + first_day_of_week * 96) // 96 % 7)
    klines_df['week'] = week_list

    # Save dataframe to CSV file
    klines_df.to_csv(ff + '_期貨.csv', index=False)


#### Binance現貨資料下載

In [ ]:
all_symbols = ['LINKUSDT','FTMUSDT','RUNEUSDT','UNIUSDT','GRTUSDT',
               'RSRUSDT','TRBUSDT','MKBUSDT','AAVEUSDT','THETAUSDT',
               'CRVUSDT','RAYUSDT','ZRXUSDT','SNXUSDT','XTZUSDT',]

for ss in all_symbols:
    # config_logging(logging, logging.DEBUG)
    Spot_client = Spot(timeout = 5)

    start_time_date = datetime(2022,1,1)
    end_time_date = datetime(2024,5,10)
    klines_raw = []

    while start_time_date < end_time_date:
        start_time_ts = datetime.timestamp(start_time_date)
        klines_raw = klines_raw + Spot_client.klines(symbol = ss, interval = "4h", startTime = int(start_time_ts)*1000, limit = 500)
        start_time_date = start_time_date + timedelta(hours=4*500)
        time.sleep(30)

    # preprocessing 1
    headers = ['open_time', 'open', 'high', 'low', 'close', 'volume', 'close_time', 'quote_vol', '#trade', 'taker_buy_vol', 'taker_buy_quote_vol', 'ignore']
    klines_df = pd.DataFrame(klines_raw, columns = headers, dtype=float)
    klines_df.drop(columns=['ignore', 'quote_vol', 'taker_buy_quote_vol'], inplace=True)

    # Convert timestamps to datetime objects
    klines_df['open_time'] = list(map(lambda x: datetime.fromtimestamp(x/1000),klines_df['open_time']))
    klines_df['close_time'] = list(map(lambda x: datetime.fromtimestamp(x/1000),klines_df['close_time']))
    # preprocessing 4
    first_day_of_week = 3 # the day of the week of the very first day of the interval
    week_list = list((klines_df.index + first_day_of_week * 96) // 96 % 7)
    klines_df['week'] = week_list

    # Save dataframe to CSV file
    klines_df.to_csv(ss + '_現貨.csv', index=False)

#### Binance 資金費率下載

In [ ]:
# 填入你的 Binance API key 和 secret
api_key = 'SOvKFyvNT6rC6OZvJ569D8LxUCWW4PVQOliJKfZ0mfRC0R85W6KZVQnAEgzNWL9e'
api_secret = 'OvtqqZCslAbfQvmt8SjrgASaJawebaNYa7sIgPBkFiBNCaiTS5RdAzXW7jR9OMnL'

client = Client(api_key, api_secret)

# 定義時間區段
start_str = '2022-01-01'
end_str = '2024-05-17'
start_date = datetime.strptime(start_str, '%Y-%m-%d')
end_date = datetime.strptime(end_str, '%Y-%m-%d')

# 定義所需要的幣種
symbols = ['LINKUSDT','FTMUSDT','RUNEUSDT','UNIUSDT','GRTUSDT',
               'RSRUSDT','TRBUSDT','MKBUSDT','AAVEUSDT','THETAUSDT',
               'CRVUSDT','RAYUSDT','ZRXUSDT','SNXUSDT','XTZUSDT',]

# 定義每次請求所需要的時間段(避免一次要求太多)
time_interval = timedelta(days=15)

# 獲取數據並存到csv檔案裡面
for symbol in symbols:
    funding_rates = []
    current_start = start_date
    while current_start < end_date:
        current_end = min(current_start + time_interval, end_date)
        funding_rate_data = client.futures_funding_rate(
            symbol=symbol,
            startTime=int(current_start.timestamp() * 1000),
            endTime=int(current_end.timestamp() * 1000)
        )
        funding_rates.extend(funding_rate_data)
        current_start = current_end
    
    # 保存數據到各自的csv檔案
    csv_file = f'{symbol}_funding_rates.csv'
    with open(csv_file, mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(['open_time', 'fundingRate'])
        for rate in funding_rates:
            # 轉換時間為UTC+0
            funding_time_utc = datetime.utcfromtimestamp(rate['fundingTime'] / 1000).strftime('%Y/%m/%d %H:%M')
            writer.writerow([funding_time_utc, rate['fundingRate']])
    
    print(f"{symbol} 資金費率數據已保存到 {csv_file}")